# Duckify Simulation notebook

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np

# Make sure ur3e-control is on the path
sys.path.insert(0, str(Path.cwd().parents[1]))

from URBasic import Joint6D, TCP6D
from URBasic import ISCoin

from duckify_simulation.duckify_sim import DuckifySim

## Connect to the simulator / the robot

In [ ]:
# Note: The simulation will always run the simulation
#       This only disables the real robot
SIMULATION = True

In [ ]:
# Connect to the simulator
duckify_sim = DuckifySim()
print("Connected to simulator")

# Create a new ISCoin object
# UR3e1 IP (closest to window): 10.30.5.158
# UR3e2 IP: 10.30.5.159
if not SIMULATION:
    iscoin = ISCoin(host="10.30.5.158", opened_gripper_size_mm=40)

In [ ]:
robot_sim = duckify_sim.robot_control
if not SIMULATION:
    robot_true = iscoin.robot_control

## Default home position

This joint position is equivalent to this one:
```txt
home_tcp = TCP6D.createFromMetersRadians(
      0.0,      # x
     -0.35,     # y
      0.20,     # z
      0.0,      # rx
      3.14,      # ry
      0.0       # rz
  )
```



In [ ]:
home = Joint6D.createFromRadians(1.1859, -1.4452, 1.2389, -1.3639, -1.5693, -0.3849)

In [ ]:
robot_sim.movej(home)
if SIMULATION:
    tcp_home = robot_sim.get_actual_tcp_pose()

In [ ]:
if not SIMULATION:
    robot_true.movej(home)
    tcp_home = robot_true.get_actual_tcp_pose()

In [ ]:
print(f"Home TCP:\n{tcp_home}")

## Set TCP

### Collect data points for calibration

In [ ]:
if not SIMULATION:
    from src.calibration import collect_data
    NUM_MEASURES = 20

    tcps = collect_data(robot_true, NUM_MEASURES)
else:
    # True tcp points
    tcps = [
        [0.09923226380963619, -0.3936237333456494, 0.2241707102377116, 0.576534502783353, -2.801623474567781, -0.4841337776020336],
        [-0.05415645689244093, -0.3953984087553033, 0.21928851997659993, 0.23402931706237418, 2.778033572687844, 0.45840199634357587],
        [-0.054524138620445695, -0.39546163267180845, 0.2194560899784096, 0.2335390262119786, 2.783785824873644, 0.4510348107203675],
        [0.0752223292033933, -0.40520983496228175, 0.2260599738507684, 0.4572457692623414, -2.9109062545914886, -0.5364679269112991],
        [0.0829983763105315, -0.3145440306229818, 0.23651078628355368, 0.3951666302237936, -2.8778640875293946, 0.020826760853426392],
        [0.17537792453011916, -0.27966304416915055, 0.1871802429879547, 0.4399208472865239, -2.4166507384311493, 0.11158336986030104],
        [-0.11994350407194002, -0.2833976738613014, 0.18952203929656486, -0.11587882239515394, 2.488783491737581, -0.2874871978684851],
        [0.042587583213620814, -0.42502159418902, 0.2219296443030877, -1.819892078036102, 2.2285954928177354, 0.5297978049692356],
        [0.04244370933942005, -0.42547028481239557, 0.22238439939526014, -1.8232450157901965, 2.2322450410810823, 0.5277815763416596],
        [0.020874413390515264, -0.34192399335641094, 0.2416659479631329, -1.6019421625827266, 2.6368378551232445, 0.06821140243218453],
        [0.020803283214023204, -0.3418580959196876, 0.24363220771625155, -1.6034030755162376, 2.6412352323485133, 0.06264503078869965],
        [-0.0790726920318296, -0.26904313811943686, 0.21128926726360914, -1.5548308665720385, 2.3598614347530305, -0.6531326432466081],
        [0.0427270175708247, -0.2473905824164213, 0.2299557666201055, 2.1997240823524864, -1.7930464879794845, 0.21897511307224746],
        [0.13236126549367827, -0.37709210594126424, 0.2134888319694367, 2.113428037038218, -2.0042534442643, -0.709899151721018],
        [0.053381366331223756, -0.3159752956877128, 0.24229245067174737, 1.8209996559857016, -2.4222938552259685, -0.050512659771693856],
        [-0.049955483412195634, -0.41217334754184276, 0.21506155931685994, -1.6191731898646298, 2.109641966964828, 0.12393449118187727],
        [0.11965373370731985, -0.426189467377058, 0.20111114739956754, -2.3836120944741555, 1.6242241381287574, 0.8568802013152502],
        [0.1072294696915219, -0.2674062588065899, 0.22084409245177472, 2.144447557404073, -1.698541221842688, -0.16060169509460684],
        [-0.10731169114667957, -0.19413670798855612, 0.15654354818487162, 2.3277646358217945, -1.2822755883172325, 1.191980512607587],
        [0.020926595559954138, -0.3139355562333332, 0.24095599082082952, 3.0691506381776072, -0.3992164073191759, 0.04457895483726155]
    ]

In [ ]:
if not SIMULATION:
    print(tcps)

{
	"tcp_calibration":[[0.09923226380963619, -0.3936237333456494, 0.2241707102377116, 0.576534502783353, -2.801623474567781, -0.4841337776020336], [-0.05415645689244093, -0.3953984087553033, 0.21928851997659993, 0.23402931706237418, 2.778033572687844, 0.45840199634357587], [-0.054524138620445695, -0.39546163267180845, 0.2194560899784096, 0.2335390262119786, 2.783785824873644, 0.4510348107203675], [0.0752223292033933, -0.40520983496228175, 0.2260599738507684, 0.4572457692623414, -2.9109062545914886, -0.5364679269112991], [0.0829983763105315, -0.3145440306229818, 0.23651078628355368, 0.3951666302237936, -2.8778640875293946, 0.020826760853426392], [0.17537792453011916, -0.27966304416915055, 0.1871802429879547, 0.4399208472865239, -2.4166507384311493, 0.11158336986030104], [-0.11994350407194002, -0.2833976738613014, 0.18952203929656486, -0.11587882239515394, 2.488783491737581, -0.2874871978684851], [0.042587583213620814, -0.42502159418902, 0.2219296443030877, -1.819892078036102, 2.2285954928177354, 0.5297978049692356], [0.04244370933942005, -0.42547028481239557, 0.22238439939526014, -1.8232450157901965, 2.2322450410810823, 0.5277815763416596], [0.020874413390515264, -0.34192399335641094, 0.2416659479631329, -1.6019421625827266, 2.6368378551232445, 0.06821140243218453], [0.020803283214023204, -0.3418580959196876, 0.24363220771625155, -1.6034030755162376, 2.6412352323485133, 0.06264503078869965], [-0.0790726920318296, -0.26904313811943686, 0.21128926726360914, -1.5548308665720385, 2.3598614347530305, -0.6531326432466081], [0.0427270175708247, -0.2473905824164213, 0.2299557666201055, 2.1997240823524864, -1.7930464879794845, 0.21897511307224746], [0.13236126549367827, -0.37709210594126424, 0.2134888319694367, 2.113428037038218, -2.0042534442643, -0.709899151721018], [0.053381366331223756, -0.3159752956877128, 0.24229245067174737, 1.8209996559857016, -2.4222938552259685, -0.050512659771693856], [-0.049955483412195634, -0.41217334754184276, 0.21506155931685994, -1.6191731898646298, 2.109641966964828, 0.12393449118187727], [0.11965373370731985, -0.426189467377058, 0.20111114739956754, -2.3836120944741555, 1.6242241381287574, 0.8568802013152502], [0.1072294696915219, -0.2674062588065899, 0.22084409245177472, 2.144447557404073, -1.698541221842688, -0.16060169509460684], [-0.10731169114667957, -0.19413670798855612, 0.15654354818487162, 2.3277646358217945, -1.2822755883172325, 1.191980512607587], [0.020926595559954138, -0.3139355562333332, 0.24095599082082952, 3.0691506381776072, -0.3992164073191759, 0.04457895483726155]]
}

### Compute and validate new TCP offset

In [ ]:
from src.calibration import get_tcp_offset, validate_calibration
tcp_offset = get_tcp_offset(tcps)
robot_sim.set_tcp(tcp_offset)
robot_sim.set_model_correction(TCP6D.createFromMetersRadians(0, 0, 0.1638, 0, 0, 0))
if not SIMULATION:
    robot_true.set_tcp(tcp_offset)

In [ ]:
motion_val_cal = validate_calibration(robot_sim)

In [ ]:
for m in motion_val_cal:
    robot_sim.movel(m)

In [ ]:
if not SIMULATION:
    robot_true.freedrive_mode()
    input("Put the robot in position for validate calibration.")
    robot_true.end_freedrive_mode()

    motion_val_cal = validate_calibration(robot_true)

    input("Ready to move?")
    for m in motion_val_cal:
        robot_true.movel(m)

## Test accuracy

In [ ]:
# MEASURED TEST CARTESIAN POSITION ON REAL ROBOT:
# TCP6D([-0.08607243040435382, -0.31652067645872695, -0.01145586981217811, 0.3950286753506294, -3.0978815742747186, -0.018834692394685076])

In [ ]:
# MEASURED TEST JOINT POSITION ON REAL ROBOT:
# Joint6D([0.9133668541908264, -1.5068980169347306, 1.8483932654010218, -1.9195991955199183, -1.5847447554217737, -0.91443378130068])
test_accuracy_pos = Joint6D.createFromRadians(0.9133668541908264, -1.5068980169347306, 1.8483932654010218, -1.9195991955199183, -1.5847447554217737, -0.91443378130068)

In [ ]:
robot_sim.movej(test_accuracy_pos)

In [ ]:
robot_sim.get_actual_joint_positions()

In [ ]:
robot_sim.get_actual_tcp_pose()

In [ ]:
robot_sim.get_all_ik_solutions(robot_sim.get_actual_tcp_pose())

In [ ]:
p = robot_sim.get_actual_tcp_pose()
p_l = p.toList()
score = 0
for joint6D in robot_sim.get_all_ik_solutions(p):
    f = robot_sim.get_fk(joint6D)
    for i, val in enumerate(f.toList()):
        score += np.abs(p_l[i]-val)
print("Accuracy score:", score/p.length())

## Compute Transformation object2robot

In [ ]:
file_path = "duckify_simulation/paths/trapezoid_letters-trapezoid_letters-trace.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
print(data["generated_at"])

In [ ]:
from src.transformation import collect_data as collect_data_tf

p_world = np.array(data["calibration"])

if len(p_world) < 4:
    print("Missing point")

if not SIMULATION:
    p_tcp = collect_data_tf(robot_true, p_world)
else:
    p_tcp = np.array([
        [-2.46780962e-02, -2.73070621e-01,  3.72949143e-02],#,  7.28589470e-01, 2.55660028e+00,  3.67059431e-01],
        [-2.36625898e-02, -3.11057752e-01,  3.73873553e-02],#,  7.40850918e-01, 2.49226370e+00,  2.76853978e-01],
        [ 3.54433603e-02, -3.08652138e-01,  3.74047430e-02],#,  3.65150801e-01, 3.09870083e+00,  2.90510505e-01],
        [ 6.26750495e-02, -2.49590824e-01,  1.60884378e-03],#,  5.31000235e-01, -2.60594978e+00, -3.50139203e-01]
    ])

In [ ]:
if not SIMULATION:
    print(p_world)
    print(p_tcp)

In [ ]:
from src.transformation import obj_to_stl, create_transformation

In [ ]:
p_world_stl = obj_to_stl(p_world)
obj2robot = create_transformation(p_world_stl, p_tcp)

## Generate trajectory (In progress)

In [ ]:
from src.transformation import tcp_trans

In [ ]:
traces = data["traces"]
draw_motions = []
move_motions = []
above = [0,0,-0.02,0,0,0]
for trace in traces:
    # Extract path
    if "face" in trace:
        normal = trace["face"]
        t_world = trace["path"]
        stl_world = [[obj_to_stl(np.array(p)), obj_to_stl(np.array(normal))] for p in t_world]
    else:
        t_world = trace["path"]
        stl_world = [[obj_to_stl(np.array(p)), obj_to_stl(np.array(n))] for p, n in t_world]

    motion = []

    # Drawing motion
    for i, p,n in enumerate(stl_world):
        p_w = (*p, *n)
        x,y,z,rx,ry,rz = obj2robot(p_w)
        
        # Add entry point above
        if i == 0:
            x_t,y_t,z_t,_,_,_ = tcp_trans([x,y,z,rx,ry,rz], above)
            m = TCP6D.createFromMetersRadians(x_t,y_t,z_t, rx,ry,rz)
            motion.append(m)

        # Add drawing point
        m = TCP6D.createFromMetersRadians(x,y,z, rx,ry,rz)
        motion.append(m)

        # Add exit point above
        if i == len(t_world)-1:
            x_t,y_t,z_t,_,_,_ = tcp_trans([x,y,z,rx,ry,rz], above)
            m = TCP6D.createFromMetersRadians(x_t,y_t,z_t, rx,ry,rz)
            motion.append(m)
    
    if motion:
        draw_motions.append(motion)
        move_motions.append((motion[0], motion[-1]))

In [ ]:
move_outside = []
for m in range(len(move_motions)):
    if m == 0:
        move_outside.append([home, move_motions[m][0]])
    else:
        move_outside.append([move_motions[m-1][-1], home, move_motions[m][0]])
move_outside.append([move_motions[-1][-1], home])

## Drawing algorithm (In progress)

### Drawing algorithm simulation

In [ ]:
import trimesh
from src.executor import draw_all_traces

obstacles = [trimesh.load("duckify_simulation/3d_objects/trapez.stl")]

In [ ]:
draw_all_traces(robot_sim, draw_motions, obstacles, home_joints=home)

### Drawing algorithm test move to position

The robot will move around the object to reach positions.

```text
                   home -> input position
output position -> home -> input position
output position -> home
```

In [ ]:
if not SIMULATION:
    robot_true.freedrive_mode()
    input("Ready to move? (Close to home position)")
    robot_true.end_freedrive_mode()

In [ ]:
if not SIMULATION:
    robot_true.movej(home)
    for k in range(len(move_motions)):
        
        # Move in position
        print(data["traces"][k]["face"])
        for m in move_outside[k]:
            if isinstance(m, TCP6D):
                robot_true.movel(m)
                print(m)
            
            elif isinstance(m, Joint6D):
                robot_true.movej(m)
                print(robot_true.get_actual_tcp_pose())
        
        print()

    print("Return home")
    for m in move_outside[-1]:
        if isinstance(m, TCP6D):
            robot_true.movel(m)
            print(m)
        elif isinstance(m, Joint6D):
            robot_true.movej(m)
            print(robot_true.get_actual_tcp_pose())

### Drawing algorithm test drawing

The robot will move around the object and draw on it.

```text
                   home -> input position -> draw
output position -> home -> input position -> draw
output position -> home
```

In [ ]:
if not SIMULATION:
    robot_true.movej(home)
    for k in range(len(move_motions)):
        
        # Move in position
        print(data["traces"][k]["face"])
        for m in move_outside[k]:
            if isinstance(m, TCP6D):
                robot_true.movel(m)
                print(m)
            
            elif isinstance(m, Joint6D):
                robot_true.movej(m)
                print(robot_true.get_actual_tcp_pose())
        
        print()

        # Draw on object
        for m in draw_motions[k]:
            robot_true.movel(m, a=0.5, v=0.1)

    print("Return home")
    for m in move_outside[-1]:
        if isinstance(m, TCP6D):
            robot_true.movel(m)
            print(m)
        elif isinstance(m, Joint6D):
            robot_true.movej(m)
            print(robot_true.get_actual_tcp_pose())